# Solutions — Utilities and Security (Hard 16)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_customers    = spark.table("samples.bakehouse.sales_customers")


## Solution 1 — String Utilities: concat_ws, ltrim, rtrim, levenshtein

In [ ]:
result_1 = (
    df_customers
    .withColumn("full_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
    .withColumn("edit_distance",
        F.levenshtein(F.col("first_name"), F.col("last_name"))
    )
    .select("customerID", "full_name", "edit_distance")
)


## Solution 2 — Hashing for Data Masking: md5, sha2, hash, crc32

In [ ]:
result_2 = (
    df_customers
    .withColumn("email_md5",    F.md5(F.col("email_address")))
    .withColumn("email_sha256", F.sha2(F.col("email_address"), 256))
    .withColumn("email_hash",   F.hash(F.col("email_address")))
    .withColumn("email_crc32",  F.crc32(F.col("email_address")))
    .select("customerID", "email_md5", "email_sha256", "email_hash", "email_crc32")
)


## Solution 3 — Encoding: base64 and unbase64

In [ ]:
test_data = [("hello world",), ("pyspark base64",), ("bakehouse",)]
test_df = spark.createDataFrame(test_data, ["text"])

result_3 = (
    test_df
    .withColumn("b64", F.base64(F.encode(F.col("text"), "UTF-8")))
    .withColumn("decoded", F.decode(F.unbase64(F.col("b64")), "UTF-8"))
    .select("text", "b64", "decoded")
)


## Solution 4 — Row Metadata: monotonically_increasing_id and spark_partition_id

In [ ]:
result_4 = (
    df_transactions
    .withColumn("row_id",      F.monotonically_increasing_id())
    .withColumn("partition_id", F.spark_partition_id())
    .select("transactionID", "row_id", "partition_id")
    .limit(100)
)


## Solution 5 — Temporary Views: createOrReplaceTempView

In [ ]:
df_transactions.createOrReplaceTempView("transactions_view")
result_5 = spark.sql("SELECT COUNT(*) AS total_rows FROM transactions_view")


## Solution 6 — crossJoin, exceptAll, intersectAll

In [ ]:
# Small samples for cross join to avoid explosion
products  = df_transactions.select("product").distinct().limit(3)
payments  = df_transactions.select("paymentMethod").distinct().limit(3)

cross     = products.crossJoin(payments)
all_prod  = df_transactions.select("product")
top_prod  = df_transactions.filter(F.col("totalPrice") > 100).select("product")
except_df = all_prod.exceptAll(top_prod)
inter_df  = all_prod.intersectAll(top_prod)

result_6 = cross.union(
    except_df.select(F.col("product"), F.lit("N/A").alias("paymentMethod"))
).union(
    inter_df.select(F.col("product"), F.lit("N/A").alias("paymentMethod"))
).limit(9)


## Solution 7 — mapInPandas

In [ ]:
import pandas as pd

def normalize_price(iterator):
    for pdf in iterator:
        min_val = pdf["totalPrice"].min()
        max_val = pdf["totalPrice"].max()
        range_val = max_val - min_val
        if range_val == 0:
            pdf["normalized_price"] = 0.0
        else:
            pdf["normalized_price"] = (pdf["totalPrice"] - min_val) / range_val
        yield pdf[["transactionID", "franchiseID", "totalPrice", "normalized_price"]]

schema = T.StructType([
    T.StructField("transactionID", T.StringType(), True),
    T.StructField("franchiseID", T.StringType(), True),
    T.StructField("totalPrice", T.DoubleType(), True),
    T.StructField("normalized_price", T.DoubleType(), True),
])

result_7 = df_transactions.select("transactionID", "franchiseID", "totalPrice").mapInPandas(normalize_price, schema=schema)


## Solution 8 — Math Utilities and repartitionByRange

In [ ]:
result_8 = (
    df_transactions
    .withColumn("sqrt_price",    F.round(F.sqrt(F.col("totalPrice")), 4))
    .withColumn("sign_price",    F.signum(F.col("totalPrice") - F.avg("totalPrice").over(Window.orderBy(F.lit(1)))))
    .withColumn("nanvl_example", F.nanvl(F.col("totalPrice").cast("double"), F.lit(0.0)))
    .repartitionByRange(4, "totalPrice")
    .select("transactionID", "totalPrice", "sqrt_price", "sign_price", "nanvl_example")
)
